O problema na selecção dos parâmetros de um esquema criptográfico é que é tipicamente um equilíbrio difícil entre usabilidade (e.g. eficiência) e segurança. Em particular, o esquema com o novo conjunto de parâmetros é claramente inseguro -- existem algoritmos capazes de recuperar a chave secreta a partir da chave pública. De facto, seria necessário aumentar consideravelmente o nível de ruído para ultrapassar essa capacidade, mas nesse caso aumentaríamos também a taxa de erro para muito próximo dos 
100
, que obviamente tornariam o esquema completamente inútil. Simplificando, podemos observar os seguintes impactos quando alteramos algum dos parâmetros:

Parâmetro	Impacto na Segurança	Impacto na taxa de Erro
aumentar 
n
aumenta	aumenta
aumentar ruído (
σ
)	aumenta	aumenta
aumentar módulo (
q
)	diminui	diminui
O objectivo é conseguir-se um nível elevado de segurança, mantendo a taxa de erro tão baixa quanto possível. Mas isso obriga a encontrar compromissos entre os diferentes parâmetros envolvidos.

[Tarefa] Impacto do ruído
Ajuste a implementação do miniPKE para usar a distribuição de ruído do esquema original (T_CHI640). Verifique e comente os impactos na correcção do algoritmo.

In [48]:
n=64
q=2^(10)
n_=m=8
B=2

In [49]:
def genMat(seedA,q,n):
    R = IntegerModRing(q)
    set_random_seed(seedA)
    A = random_matrix(R,n,n)
    return A

In [50]:
T_CHI640 = [4643, 13363, 20579, 25843, 29227, 31145, 32103, 32525, 32689, 32745, 32762, 32767]
def chiSample(linhas,colunas,q,tabela = T_CHI640):
    R = ZZ
    matriz = matrix(R,linhas,colunas)
    for l in range(linhas):
        for c in range(colunas):
            x = randint(0, 2^(15)-1)
            i = 0
            while i < len(tabela):
                if tabela[i] > x:
                    break
                i+=1

            bit_sinal = randint(0,1)
            if bit_sinal:
                s = -1
            else:
                s = 1;
            matriz[l, c] = s * i
    return matriz

In [51]:
def encode (bits, n, enq, B = 2): 
    assert len(bits) == n * n * B

    R = Integers(q)
    scale = q // (2^B)

    M = Matrix(R, n, n)

    idx = 0
    for i in range(n):
        for j in range(n):
            # pega B bits
            val = 0
            for b in range(B):
                val = (val << 1) | bits[idx]
                idx += 1

            M[i, j] = R(val * scale)

    return M

In [52]:
def decode (M, q , B = 2):
    n = M.nrows()
    bits = []

    for i in range(n):
        for j in range(n):
            x = Integer(M[i, j])

            val = round(x * (2^B) / q)

            val = val % (2^B)

            for b in reversed(range(B)):
                bits.append((val >> b) & 1)

    return bits

In [53]:
def key_Gen(q,table,n,n_,):
    R = IntegerModRing(q)
    seedA= randint(0,q-1)
    A=genMat(seedA,q,n)
    S = chiSample(n, n_, table)
    E = chiSample(n, n_, table)
    B_pk=A*S+E
    pk=(seedA,B_pk)
    sk=S
    return (sk,pk)

In [54]:
def enc(pk,m,table,n,n_,q):
    R = IntegerModRing(q)
    seedA,B_pk=pk
    A=genMat(seedA,q,n)
    S_=chiSample(n_,n,table)
    E_=chiSample(n_,n,table)
    E__=chiSample(n_,n_,table)
    c1=S_*A+E_
    V_=S_*B_pk+E__
    M = encode(m, n_, q)
    c2=V_+M
    return(c1,c2)

In [55]:
def dec(sk,C,pk,q,n):
    c1,c2=C
    S=sk
    seedA,B_pk=pk
    A=genMat(seedA,q,n)
    V= c1*S
    M=c2-V
    m_=decode(M,q)
    return m_

In [56]:
m_bits = [randint(0, 1) for _ in range(n_ * n_ * B)]

matriz_inicial=encode(m_bits, n_, q, B)
show(matriz_inicial)
sk, pk = key_Gen(q, T_CHI640, n, n_)
C = enc(pk, m_bits, T_CHI640, n, n_, q)
m_ = dec(sk, C, pk, q, n)


c1, c2 = C
V = c1 * sk
matriz_final = c2 - V
show(matriz_final)





[512 768   0   0   0 256 768 256]
[768 512 512 512 512 512 768 256]
[256 256 256 768   0 512 256 256]
[512   0 768 768 256   0 256   0]
[512 768 768 768 768 256 256 768]
[  0 512 512 512 512 768 256 512]
[512 768 256   0 256 512   0 256]
[512 256   0   0 256 512 256 256]

[ 345  760 1005  873  870  301  766  310]
[ 648  529  433  494  555  601  777  128]
[ 283  210  262  963  829  545  269  340]
[ 445 1013  779  721  164   28  143   88]
[ 478  777  675  820  662  338  184  965]
[1010  555  487  617  628  615  178  639]
[ 458  724  285   93  379  602 1019  316]
[ 510  367  990  894   71  614  263  315]

In [57]:
m_bits

[1,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 1]

In [58]:
m_

[0,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 1]

In [59]:
print(m_bits == m_)

count=0
for i in range(len(m_bits)):
    if m_bits[i] != m_[i]:
        count+=1


print(count)

False
17
